In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

if os.environ['Anthropic_API_KEY']:
    print("API KEY is set")

API KEY is set


In [2]:
from langchain_anthropic import ChatAnthropic


d:\RAG_POC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm=ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

##  Rag Implementation with PDF


In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "./Docs/fabric-admin.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

docs

[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft Learn', 'creationdate': '2025-12-09T19:42:08+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2025-12-09T19:42:08+00:00', 'source': './Docs/fabric-admin.pdf', 'total_pages': 290, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft 

## Creating own Metadata for PDF Chunk

In [5]:
for i in docs:

    i.metadata = {"source": "fabric-admin.pdf",
                  "developer": "Microsoft"}

#### Step2 : Splitting the Document into CHUNK

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Configure notifications\nSet up metadata scanning\nEnable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCE

In [17]:
## size of chunks
len(chunks)

652

In [7]:
chunks[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}

#### Step3: Creating Embedding for the Chunks

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

## HuggingFace is Free
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2")

## Claude Doesn't Have Its Own Embedding Model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2630.57it/s]


#### Step4:Store Embeddings in Existing Local Vector Store

In [10]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="./VectorDB/",
                     embedding_function=embedding_model)

In [11]:
vectorstore.add_documents(chunks)

['28357710-2041-4c00-8c05-31823086f402',
 'e35cddc2-09bf-41dd-934d-b1936e43ca4c',
 'f34c4b5d-66d7-418b-932d-664822f985a1',
 'f42ef0bc-3ada-4acb-8973-912896e0009d',
 '4f0edcd5-2459-4ced-9724-c5702662ddde',
 '74fa6088-971f-4a20-b30c-d86600998ba1',
 'b37ad33e-1c8b-4af1-badc-8b1f6c61d228',
 '55a9243a-d90a-450a-9b87-1caf952e74ae',
 '7d535cc4-a55b-4e69-8788-dda931ee3b07',
 '12c8cd14-6d33-466c-b1f3-98b403e7bd4d',
 '98a099ca-2f72-45ff-8fa8-84e02f5830c6',
 'd60a1414-6794-4ef6-b858-156f9431b939',
 '119486c3-b0bb-49f8-b4d7-40f5dec7fad8',
 '97eea394-fd35-4b94-b254-6fe4b4f1275c',
 '4ded3120-c707-4317-8959-a77d6da21972',
 'c392bc1d-ed73-49a6-a5c5-6108b95e597c',
 'f8150c7c-9ccc-4798-acc6-5d3898535d4c',
 '1f6db43d-e638-48be-a2bd-b8c27f4cb737',
 '4ea91601-0215-443c-8842-c37ce9771d3c',
 '5c12acb2-633f-4781-866d-6c89d8f5f0e9',
 'dc060bf1-dae2-494c-b6d7-bbf36daf70f8',
 '0fc62daf-5335-4bec-905a-63b2ce24cadf',
 'b6766c81-7ffb-4df7-a137-c5a7145a1afd',
 '868680fa-98e7-40c0-b0fa-8fca93466ac6',
 'c6b45fd2-a38f-

In [17]:
vectorstore.similarity_search("Why is MS Fabric admin?", k= 3)

[Document(metadata={'developer': 'Microsoft', 'source': 'fabric-admin.pdf'}, page_content="What is the admin portal?\nThe Microsoft Fabric admin portal includes settings that govern Microsoft Fabric. For example,\nyou can make changes to tenant settings, access the Microsoft 365 admin portal, and control\nhow users interact with Microsoft Fabric.\nTo access the admin portal, you need a Fabric license and the Fabric administrator role.\nIf you're not in one of these roles, you only see Capacity settings in the admin portal.\nThe many controls in the admin portal are listed in the following table, with links to relevant\ndocumentation for each one.\nFeature Description\nTenant settings Enable, disable, and configure Microsoft Fabric.\nUsers Manage users in the Microsoft 365 admin portal.\nPremium Per User Configure auto refresh and semantic model workload settings.\nAudit logs Audit Microsoft Fabric activities in the Microsoft Purview portal.\nDomains Manage and organize business data us

In [18]:
response = llm.invoke("What is MS Fabric admin?")
print(response.content)

# MS Fabric Admin

MS Fabric Admin refers to the **administrative capabilities and tools** within **Microsoft Fabric**, Microsoft's unified analytics platform. Here are the key aspects:

## Core Responsibilities

- **User & Access Management**: Control who can access Fabric resources and assign permissions
- **Capacity Management**: Allocate and monitor compute resources (Fabric capacity)
- **Workspace Administration**: Create, manage, and organize workspaces
- **Governance & Security**: Enforce organizational policies and data protection standards
- **Monitoring & Auditing**: Track usage, performance, and compliance

## Key Admin Features

| Feature | Purpose |
|---------|---------|
| **Admin Portal** | Centralized dashboard for managing Fabric settings |
| **Capacity Management** | Monitor and allocate compute resources |
| **Tenant Settings** | Configure organization-wide policies |
| **Audit Logs** | Track user activities and changes |
| **Data Governance** | Manage data lineage, s